In [ ]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *
from _fractures import *
from _initial_conditions import *


# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

np.random.seed(0)

If you want try out your own grid parameters, uncommment the code in red and set your own parameters. Do not change the 4th line that defines `dx, dy, dz`, as these values are purely determined from other set values.

In [ ]:
# -- Grid parameters --

# -- Lower/Upper Bound, Number of Points, Separation Between Points --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

# manually setting if you want to try other things
'''
xmin,ymin,zmin = -1,-1,-1
xmax,ymax,zmax = 1,1,1
Nx, Ny, Nz = 8, 8, 8
dx, dy, dz = (xmax-xmin)/(Nx-1), (ymax-ymin)/(Ny-1), (zmax-zmin)/(Nz-1)
'''

# -- Staggered Grid Points for each wavefield component of seismic wave equation w = [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

Only change `file_label_2`, which controls the initial condition (Ricker or Gaussian), and `file_label_5`, which controls fracture geometry.

In [ ]:
## Configure which physical scenario to run for the notebook, times will be determined later
#Decide if quantum result will be calculated (takes a long time to run, don't run with over 6x6x6 case
calc_quantum = False


#If quantum is true, calculate quantum and classical, if false just calculate classical
file_label_1 = 'classical'
if calc_quantum:
    file_label_1='quantum'

#initial conditions: ricker_IC_1 or gaussian_IC_1
file_label_2 = 'gaussian_IC_1'

#Use grid size to create this label 
file_label_3 = str(Nx)+'x'+str(Ny)+'x'+str(Nz)

#Use physical domain for this label

file_label_4 = str(xmin)+'to'+str(xmax)

#Fracture geometry 
# options: two_perp_xy_yz, no_fracture, one_xy, one_yz
file_label_5 = 'two_perp_xy_yz'

In [ ]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture)

#Set fracture thickness
fracture_thickness = 0
#Set density and compliance matrix according to fracture geometry specified earlier
if file_label_5 == 'two_perp_xy_yz':
    rho_model, S_matrix = two_perpendicular_fractures(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'no_fracture':
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)
elif file_label_5 == 'one_xy':
    rho_model, S_matrix = one_horizontal_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'one_yz':
    rho_model, S_matrix = one_vertical_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
else:
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
print("Number of grid points: ", psi_len)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)

In [ ]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

#phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
if file_label_2 == 'ricker_IC_1':
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
elif file_label_2 == 'gaussian_IC_1':
    phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
else:
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
    
# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])
# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

In [ ]:
# Time Integration (Has classical integration errors (DOP853))
def get_classical_state_vector(t):
    #phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='DOP853').y.T[-1] 
    phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='RK45').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical

Time ranges include `[5e-6,1e-4], [5e-5,1e-3],[.005,.1],[.05,1],[.25,5],[.5,10]`

I found that only the first time range makes sense, as the other time ranges get chaotic too quickly.

The `np.linspace` below is structured as `np.linspace(t_start, t_end, n_timesteps)`

In [ ]:
history = []
for t in np.linspace(5e-6,1e-4,20):
    history.append(get_classical_state_vector(t))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def plot_time_slider2(k_fixed, history, xmin, ymin, zmin, xmax, ymax, zmax, Nx, Ny, Nz, dx, dy, dz):
    # Safety check for k
    k_fixed = max(0, min(k_fixed, Nz - 1))
    n_steps = len(history)
    
    # Calculate the physical z-coordinate for the title
    z_phys = zmin + k_fixed * dz
    
    # Define the physical boundaries for the 2D x-y plane plots
    plot_extent = [xmin, xmax, ymin, ymax]

    frames = []
    
    # Track global max values for consistent colorbars
    global_max = {
        'v_x': 0, 'v_y': 0, 'v_z': 0, 'v_mag': 0,
        'sigma_xx': 0, 'sigma_yy': 0, 'sigma_zz': 0,
        'sigma_xy': 0, 'sigma_xz': 0, 'sigma_yz': 0
    }

    # Grid sizes
    N_vx = (Nx - 1) * Ny * Nz
    N_vy = Nx * (Ny - 1) * Nz
    N_vz = Nx * Ny * (Nz - 1)
    N_main = Nx * Ny * Nz
    N_sxy = (Nx - 1) * (Ny - 1) * Nz
    N_sxz = (Nx - 1) * Ny * (Nz - 1)
    N_syz = Nx * (Ny - 1) * (Nz - 1)

    for t in range(n_steps):
        phi = history[t]
        idx = 0
        
        # --- Unpacking ---
        vx_raw = phi[idx:idx+N_vx].reshape(Nz, Ny, Nx-1); idx += N_vx
        vy_raw = phi[idx:idx+N_vy].reshape(Nz, Ny-1, Nx); idx += N_vy
        vz_raw = phi[idx:idx+N_vz].reshape(Nz-1, Ny, Nx); idx += N_vz
        
        sxx = phi[idx:idx+N_main].reshape(Nz, Ny, Nx); idx += N_main
        syy = phi[idx:idx+N_main].reshape(Nz, Ny, Nx); idx += N_main
        szz = phi[idx:idx+N_main].reshape(Nz, Ny, Nx); idx += N_main
        
        sxy_raw = phi[idx:idx+N_sxy].reshape(Nz, Ny-1, Nx-1); idx += N_sxy
        sxz_raw = phi[idx:idx+N_sxz].reshape(Nz-1, Ny, Nx-1); idx += N_sxz
        syz_raw = phi[idx:idx+N_syz].reshape(Nz-1, Ny-1, Nx)

        # --- Interpolation & Slicing ---
        
        # v_x (Staggered X)
        vx_slice = np.zeros((Ny, Nx))
        vx_slice[:, 0] = vx_raw[k_fixed, :, 0]
        vx_slice[:, -1] = vx_raw[k_fixed, :, -1]
        vx_slice[:, 1:-1] = 0.5 * (vx_raw[k_fixed, :, :-1] + vx_raw[k_fixed, :, 1:])
        
        # v_y (Staggered Y)
        vy_slice = np.zeros((Ny, Nx))
        vy_slice[0, :] = vy_raw[k_fixed, 0, :]
        vy_slice[-1, :] = vy_raw[k_fixed, -1, :]
        vy_slice[1:-1, :] = 0.5 * (vy_raw[k_fixed, :-1, :] + vy_raw[k_fixed, 1:, :])
        
        # v_z (Staggered Z)
        if 0 < k_fixed < Nz - 1:
             vz_slice = 0.5 * (vz_raw[k_fixed-1, :, :] + vz_raw[k_fixed, :, :])
        elif k_fixed == 0:
             vz_slice = vz_raw[0, :, :]
        else:
             vz_slice = vz_raw[-1, :, :]
             
        v_mag = np.sqrt(vx_slice**2 + vy_slice**2 + vz_slice**2)
        
        # Normal Stresses
        sxx_slice = sxx[k_fixed, :, :]
        syy_slice = syy[k_fixed, :, :]
        szz_slice = szz[k_fixed, :, :]
        
        # --- SHEAR STRESSES CORRECTION ---
        
        # 1. Sigma XY
        sxy_slice = np.zeros((Ny, Nx))
        sxy_slice[1:, 1:] = sxy_raw[k_fixed, :, :]
        
        # 2. Sigma XZ
        sxz_slice = np.zeros((Ny, Nx))
        if k_fixed < Nz-1:
            sxz_slice[:, 1:] = sxz_raw[k_fixed, :, :]
            
        # 3. Sigma YZ
        syz_slice = np.zeros((Ny, Nx))
        if k_fixed < Nz-1:
            syz_slice[1:, :] = syz_raw[k_fixed, :, :]

        frame_data = {
            'v_x': vx_slice, 'v_y': vy_slice, 'v_z': vz_slice, 'v_mag': v_mag,
            'sigma_xx': sxx_slice, 'sigma_yy': syy_slice, 'sigma_zz': szz_slice,
            'sigma_xy': sxy_slice, 'sigma_xz': sxz_slice, 'sigma_yz': syz_slice
        }
        frames.append(frame_data)
        
        for key in global_max:
            current_max = np.max(np.abs(frame_data[key]))
            if current_max > global_max[key]:
                global_max[key] = current_max

    # Plotting Setup
    fig, axes = plt.subplots(2, 5, figsize=(22, 9))
    title_text = fig.suptitle(f'Time Evolution at Z-Slice {z_phys:.3f} (k={k_fixed}, t=0)', fontsize=16, fontweight='bold')
    
    layout = [
        ('v_x', 'Velocity X', 'RdBu'), ('v_y', 'Velocity Y', 'RdBu'), ('v_z', 'Velocity Z', 'RdBu'), 
        ('v_mag', 'Velocity Mag', 'viridis'), ('sigma_xx', 'Normal XX', 'RdBu'),
        ('sigma_yy', 'Normal YY', 'RdBu'), ('sigma_zz', 'Normal ZZ', 'RdBu'),
        ('sigma_xy', 'Shear XY', 'RdBu'), ('sigma_xz', 'Shear XZ', 'RdBu'), ('sigma_yz', 'Shear YZ', 'RdBu')
    ]
    
    images = {}
    
    for ax, (key, title, cmap) in zip(axes.flatten(), layout):
        data = frames[0][key]
        vmax = global_max[key]
        if vmax == 0: vmax = 1e-10
        
        # Added the 'extent' parameter here to map indices to physical space
        if cmap == 'RdBu':
            im = ax.imshow(data, cmap=cmap, origin='lower', extent=plot_extent, vmin=-vmax, vmax=vmax)
        else:
            im = ax.imshow(data, cmap=cmap, origin='lower', extent=plot_extent, vmin=0, vmax=vmax)
            
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        images[key] = im
        
    plt.tight_layout()

    def update(t):
        title_text.set_text(f'Time Evolution at Z-Slice {z_phys:.3f} (k={k_fixed}, t={t})')
        for key in images:
            images[key].set_data(frames[t][key])
        return list(images.values())

    anim = FuncAnimation(fig, update, frames=n_steps, interval=200, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

The following block will create plots for each slice. If you want just a single slice, create a new code block, get rid of the for loop, and just indicate a single value for `k_fixed`, which should be an integer value in `[0, ..., Nz-1]`

In [ ]:
for i in range(Nz):
    html_widget = plot_time_slider2(k_fixed=i, history=history, 
                                    xmin=xmin, ymin=ymin, zmin=zmin, xmax=xmax, ymax=ymax, zmax=zmax, 
                                    Nx=Nx, Ny=Ny, Nz=Nz, dx=dx, dy=dy, dz=dz)
    display(html_widget)